## Transform Customer Data
### 1.Remove the Records with Null Customer_id
### 2.Reomove Extact duplicates Records
### 3.Remove the Duplicate based on created Timestamps
### 4.Cast the correct data Types
### 5.Transform to silver Schema

# Remove the records with Null Customer_id

In [0]:
%sql
SELECT *
FROM gizmobox_nara.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id ASC
LIMIT 100

# Remove the Duplicates

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct 
AS
SELECT DISTINCT *
 FROM gizmobox_nara.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id

In [0]:
%sql
with cte_max AS
(
  SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id
)
SELECT t.*
    FROM v_customers_distinct t
    INNER JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp
    

### CAST the column to correct data type

In [0]:
%sql
WITH cte_max AS
(
  SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) As created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone
    FROM v_customers_distinct t
    INNER JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp

In [0]:
%sql
DROP TABLE gizmobox_nara.silver.customers;
CREATE TABLE gizmobox_nara.silver.customers AS
WITH cte_max AS
(
  SELECT customer_id,
      MAX(created_timestamp) AS max_created_timestamp
      FROM v_customers_distinct
      GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) As created_timestamp,
    t.customer_id,
    t.customer_name,
    CAST(t.date_of_birth AS DATE) AS date_of_birth,
    t.email,
    CAST(t.member_since AS DATE) AS member_since,
    t.telephone
    FROM v_customers_distinct t
    INNER JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp

In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.customers